In [1]:
import pandas as pd
from rouge_score import rouge_scorer
from sarathi.benchmark.request_generator.real_request_generator import RealRequestGenerator
from sarathi.benchmark.config import Config

from matplotlib import pyplot as plt
import matplotlib.lines as mlines

/anaconda3/envs/vattn/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


unable to import module pod_attn


2025-06-17 23:15:51,295	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
request_generator = RealRequestGenerator(config=Config({'num_requests': 100}), prompt_length=1000)
cnn_prompts = request_generator.get_cnn_prompts()


In [3]:
scorer = rouge_scorer.RougeScorer(['rougeL', 'rouge1', 'rouge2'])

def get_rougeL_score(prompt, output):
    return scorer.score(prompt, output)['rougeL']

def get_avg_rougeL_score(prompts, outputs):
    precisions = []
    recalls = []
    fmeasures = []
    for prompt, output in zip(prompts, outputs):
        scores = get_rougeL_score(prompt, output)
        precisions.append(scores.precision)
        recalls.append(scores.recall)
        fmeasures.append(scores.fmeasure)
    return sum(precisions) / len(precisions), sum(recalls) / len(recalls), sum(fmeasures) / len(fmeasures)


In [4]:
def get_rougeL_for_df(df, prompts):
    avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(prompts, df.output)
    return avg_rouge_score

In [5]:
df = pd.read_csv("/workspace/xutingl/vattention-ee/outputs_13b_tuned/req_100_batch_4_conf09_csv/ee_batch1.csv")
get_rougeL_for_df(df, cnn_prompts)


0.2619993865835331

In [9]:
df_nobatch_copy = pd.read_csv("/workspace/xutingl/vattention-ee/outputs_13b_tuned/req_100_batch_4_conf09_csv/ee_batch1.csv")
get_rougeL_for_df(df_nobatch_copy, cnn_prompts)


0.18171258738871401

In [7]:
df = pd.read_csv("/workspace/xutingl/vattention-ee/outputs_13b_tuned/req_100_batch_4_conf09_csv/rebatching.csv")
get_rougeL_for_df(df, cnn_prompts)

0.13310735200419738

In [8]:
df_rebatching_copy = pd.read_csv("/workspace/xutingl/vattention-ee/outputs_13b_tuned/req_100_batch_4_conf09_csv/rebatching.csv")
get_rougeL_for_df(df_rebatching_copy, cnn_prompts)

0.20958217876957672

In [5]:
policies = ["ee_nobatch", "off","eager", "lazy", "average", "rebatching"]
throughputs = []
rouge_scores = []
tpot = []

num_ee_tokens = []
num_no_ee_tokens = []
avg_conf_score = []
avg_conf_score_ee = []


ee_df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_13b_tuned/req_100_batch_4_conf09_csv/ee_batch1.csv")
avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, ee_df.output)
throughputs.append(ee_df.throughput.mean())
rouge_scores.append(avg_rouge_score)

tpot.append(ee_df.tpot.values[0])
num_ee_tokens.append(ee_df.num_ee_tokens.values[0])
num_no_ee_tokens.append(ee_df.num_no_ee_tokens.values[0])
avg_conf_score.append(ee_df.avg_conf_score.values[0])
avg_conf_score_ee.append(ee_df.avg_conf_score_ee.values[0])

for policy in policies[1:]:
    df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_13b_tuned/req_100_batch_4_conf09_csv/{policy}.csv")
    avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, df.output)
    throughputs.append(df.throughput.mean())
    rouge_scores.append(avg_rouge_score)

    tpot.append(df.tpot.values[0])
    num_ee_tokens.append(df.num_ee_tokens.values[0])
    num_no_ee_tokens.append(df.num_no_ee_tokens.values[0])
    avg_conf_score.append(df.avg_conf_score.values[0])
    avg_conf_score_ee.append(df.avg_conf_score_ee.values[0])
df = pd.DataFrame({
    "policy": policies,
    "throughput": throughputs,
    "TPOT": tpot,
    "ROUGE-L": rouge_scores,
    "num_ee_tokens": num_ee_tokens,
    "num_no_ee_tokens": num_no_ee_tokens,
    "avg_conf_score": avg_conf_score,
    "avg_conf_score_ee": avg_conf_score_ee
})
df
# 06/12/2025 llama2-13b, layer=24, conf=0.9, tuned head, nocopy

,policy,throughput,TPOT,ROUGE-L,num_ee_tokens,num_no_ee_tokens,avg_conf_score,avg_conf_score_ee
0,ee_nobatch,37.376061,0.025229,0.283271,1723,15298,0.711121,0.963793
1,off,112.126392,0.008161,0.291897,0,1,0.760423,0.000000
2,eager,117.288380,0.007925,0.171372,4428,15167,0.330279,0.391746
3,lazy,110.340222,0.008305,0.291897,1,12295,0.760418,0.985473
4,average,108.240400,0.008492,0.291272,8,12623,0.758154,0.960654
5,rebatching,118.675548,0.007849,0.139193,968,20847,0.246281,0.963293


In [7]:
policies = ["ee_nobatch", "off","eager", "lazy", "average", "rebatching"]
throughputs = []
rouge_scores = []
tpot = []

num_ee_tokens = []
num_no_ee_tokens = []
avg_conf_score = []
avg_conf_score_ee = []


ee_df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_13b_tuned/req_100_batch_4_conf09_csv/ee_batch1.csv")
avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, ee_df.output)
throughputs.append(ee_df.throughput.mean())
rouge_scores.append(avg_rouge_score)

tpot.append(ee_df.tpot.values[0])
num_ee_tokens.append(ee_df.num_ee_tokens.values[0])
num_no_ee_tokens.append(ee_df.num_no_ee_tokens.values[0])
avg_conf_score.append(ee_df.avg_conf_score.values[0])
avg_conf_score_ee.append(ee_df.avg_conf_score_ee.values[0])

for policy in policies[1:]:
    df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_13b_tuned/req_100_batch_4_conf09_csv/{policy}.csv")
    avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, df.output)
    throughputs.append(df.throughput.mean())
    rouge_scores.append(avg_rouge_score)

    tpot.append(df.tpot.values[0])
    num_ee_tokens.append(df.num_ee_tokens.values[0])
    num_no_ee_tokens.append(df.num_no_ee_tokens.values[0])
    avg_conf_score.append(df.avg_conf_score.values[0])
    avg_conf_score_ee.append(df.avg_conf_score_ee.values[0])
df = pd.DataFrame({
    "policy": policies,
    "throughput": throughputs,
    "TPOT": tpot,
    "ROUGE-L": rouge_scores,
    "num_ee_tokens": num_ee_tokens,
    "num_no_ee_tokens": num_no_ee_tokens,
    "avg_conf_score": avg_conf_score,
    "avg_conf_score_ee": avg_conf_score_ee
})
df
# 06/13/2025 llama2-13b, layer=24, conf=0.9, tuned head, fixing copy

,policy,throughput,TPOT,ROUGE-L,num_ee_tokens,num_no_ee_tokens,avg_conf_score,avg_conf_score_ee
0,ee_nobatch,37.240222,0.025563,0.184049,1862,35615,0.286122,0.962251
1,off,112.126392,0.008161,0.291897,0,1,0.760423,0.000000
2,eager,116.717012,0.008014,0.124502,3948,19279,0.193645,0.372913
3,lazy,110.340222,0.008305,0.291897,1,12295,0.760418,0.985473
4,average,108.240400,0.008492,0.291272,8,12623,0.758154,0.960654
5,rebatching,120.514002,0.007726,0.203826,1241,20709,0.347705,0.961993


In [8]:
policies = ["ee_nobatch", "off","eager", "lazy", "average", "rebatching"]
throughputs = []
rouge_scores = []
tpot = []

num_ee_tokens = []
num_no_ee_tokens = []
avg_conf_score = []
avg_conf_score_ee = []


ee_df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_13b_tuned/req_100_batch_4_conf09_csv/ee_batch1.csv")
avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, ee_df.output)
throughputs.append(ee_df.throughput.mean())
rouge_scores.append(avg_rouge_score)

tpot.append(ee_df.tpot.values[0])
num_ee_tokens.append(ee_df.num_ee_tokens.values[0])
num_no_ee_tokens.append(ee_df.num_no_ee_tokens.values[0])
avg_conf_score.append(ee_df.avg_conf_score.values[0])
avg_conf_score_ee.append(ee_df.avg_conf_score_ee.values[0])

for policy in policies[1:]:
    df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_13b_tuned/req_100_batch_4_conf09_csv/{policy}.csv")
    avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, df.output)
    throughputs.append(df.throughput.mean())
    rouge_scores.append(avg_rouge_score)

    tpot.append(df.tpot.values[0])
    num_ee_tokens.append(df.num_ee_tokens.values[0])
    num_no_ee_tokens.append(df.num_no_ee_tokens.values[0])
    avg_conf_score.append(df.avg_conf_score.values[0])
    avg_conf_score_ee.append(df.avg_conf_score_ee.values[0])
df = pd.DataFrame({
    "policy": policies,
    "throughput": throughputs,
    "TPOT": tpot,
    "ROUGE-L": rouge_scores,
    "num_ee_tokens": num_ee_tokens,
    "num_no_ee_tokens": num_no_ee_tokens,
    "avg_conf_score": avg_conf_score,
    "avg_conf_score_ee": avg_conf_score_ee
})
df
# 06/13/2025 llama2-13b, layer=24, conf=0.9, tuned head, fixing copy

,policy,throughput,TPOT,ROUGE-L,num_ee_tokens,num_no_ee_tokens,avg_conf_score,avg_conf_score_ee
0,ee_nobatch,37.620516,0.025084,0.283271,1723,15298,0.711121,0.963793
1,off,112.126392,0.008161,0.291897,0,1,0.760423,0.000000
2,eager,116.717012,0.008014,0.124502,3948,19279,0.193645,0.372913
3,lazy,110.340222,0.008305,0.291897,1,12295,0.760418,0.985473
4,average,107.894579,0.008530,0.299099,11,12767,0.761219,0.970868
5,rebatching,100.129342,0.009290,0.187872,1022,19416,0.312038,0.962872
